# RUSH 4 — Segmentation des utilisateurs Instagram
## 01 · Audit des données, exploration et premières briques de scoring

**Question du département :** comment l'usage d'Instagram est-il relié au stress, au bonheur et au sommeil, et peut-on en déduire des segments de users utiles aux équipes produit ?

**Ce que fait ce notebook (et ce qu'il ne fait pas encore)**
1. Compare les deux extraits et justifie les champs retenus.
2. Nettoie et documente les anomalies détectables.
3. Explore les liens usage ↔ bien-être, y compris les **liens nuls**.
4. Construit des variables dérivées justifiées, un **score de bien-être** et une cible binaire.
5. Pose un **modèle de prédiction** et une **première segmentation**, puis mesure honnêtement ce que la segmentation ajoute au simple classement par temps passé.
6. Pose les limites statistiques et les contraintes de protection des données.

> **Règle appliquée dès ce notebook :** aucune ligne individuelle n'est affichée. Toutes les sorties sont des agrégats. Les identifiants ne servent qu'à compter.

### Cadrage : critères de réussite fixés *avant* les résultats

| Critère | Définition opérationnelle | Vérification dans le notebook |
|---|---|---|
| **Assignable** | tout utilisateur peut être placé dans un segment avec les seuls champs dont Instagram dispose pour tous | fonction `assign_cluster()` + `assign_usage_segment()` |
| **Stable** | deux échantillons différents donnent les mêmes segments | ARI split-half |
| **Distinct** | segments de taille suffisante, non triviaux | tailles minimales, silhouette |
| **Incrementiel** | apporte quelque chose par rapport au temps passé | ΔAUC, ΔR², ARI vs déciles de temps |
| **Interprétable** | peu de variables, noms compréhensibles par le produit | liste des features |
| **Sûr juridiquement** | pas de champ santé dans la segmentation, pas de ciblage des mineurs, pas de cellule < 50 | audit final |

### Protocole
- **Extrait A** = `instagram_usage_lifestyleSafe.csv` (référence pour les dates et le comportement).
- **Extrait B** = `instagram_users_lifestyleSafe.csv` (contrôle de cohérence ; utilisé uniquement pour récupérer l'éducation, voir §1).
- **Population** : 200 000 users, panneau auto-déclaré pour les variables de bien-être.
- **Graine aléatoire** : 42 partout.


## 0. Environnement et helpers

Dépendances : `pandas`, `numpy`, `matplotlib`, `scikit-learn`, `scipy`. Pas de `seaborn` requis.


In [ ]:
from pathlib import Path
import time
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    roc_auc_score, roc_curve, average_precision_score, brier_score_loss,
    silhouette_score, adjusted_rand_score, r2_score,
)
from sklearn.inspection import permutation_importance

try:
    from IPython.display import display
except ImportError:          # permet d'exécuter le notebook comme un script
    display = print

RANDOM_STATE = 42
DATA_DIR = Path(".")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
plt.rcParams.update({
    "figure.figsize": (11, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def save_fig(name: str, dpi: int = 120) -> None:
    '''Sauvegarde robuste : les dossiers synchronisés (OneDrive) peuvent verrouiller un fichier.'''
    path = (FIG_DIR / name).resolve()
    for _ in range(3):
        try:
            plt.savefig(path, dpi=dpi)
            return
        except OSError:
            time.sleep(0.5)
    plt.savefig(path, dpi=dpi)

print("Environnement prêt.")


## 1. Chargement et audit des deux extraits

On charge les deux fichiers **sans les concaténer** : ce sont deux versions du même panneau, pas deux populations.


In [ ]:
path_a = DATA_DIR / "instagram_usage_lifestyleSafe.csv"
path_b = DATA_DIR / "instagram_users_lifestyleSafe.csv"

raw_a = pd.read_csv(path_a)
raw_b = pd.read_csv(path_b)

audit = pd.DataFrame(
    {
        "Extrait A (usage)": [
            raw_a.shape[0], raw_a.shape[1], raw_a["user_id"].nunique(),
            int(raw_a.duplicated().sum()), int(raw_a.isna().sum().sum()),
        ],
        "Extrait B (users)": [
            raw_b.shape[0], raw_b.shape[1], raw_b["user_id"].nunique(),
            int(raw_b.duplicated().sum()), int(raw_b.isna().sum().sum()),
        ],
    },
    index=["Lignes", "Colonnes", "user_id uniques", "Lignes dupliquées", "Valeurs manquantes"],
)
display(audit)
print("Colonnes identiques :", list(raw_a.columns) == list(raw_b.columns))
print("Mêmes user_id, même ordre :", bool((raw_a["user_id"].values == raw_b["user_id"].values).all()))
print("Aucune valeur manquante :", raw_a.isna().sum().sum() == 0 and raw_b.isna().sum().sum() == 0)


### 1.1 Où les deux extraits divergent-ils ?

Un seul extrait doit servir de référence. On quantifie d'abord les divergences, puis on teste leur **cohérence interne** (dates, modalités, heures de travail).


In [ ]:
def divergence_profile(a: pd.DataFrame, b: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in a.columns:
        if pd.api.types.is_numeric_dtype(a[col]):
            diff = ~np.isclose(a[col].astype(float), b[col].astype(float), equal_nan=True)
            kind = "numérique"
        else:
            diff = a[col].fillna("NA").astype(str) != b[col].fillna("NA").astype(str)
            kind = "catégorielle"
        n = int(diff.sum())
        if n:
            rows.append({"Colonne": col, "Type": kind, "Lignes divergentes": n,
                         "% divergentes": round(100 * n / len(a), 2)})
    return pd.DataFrame(rows).sort_values("Lignes divergentes", ascending=False)

divergences = divergence_profile(raw_a, raw_b)
display(divergences)

# Contrôles de cohérence internes
checks = []
for name, frame in {"A": raw_a, "B": raw_b}.items():
    login_year = pd.to_datetime(frame["last_login_date"], errors="coerce").dt.year
    no_job = frame["employment_status"].isin(["Unemployed", "Not employed"])
    checks.append({
        "Extrait": name,
        "Création > année de dernière connexion": int((frame["account_creation_year"] > login_year).sum()),
        "Création en 2026": int((frame["account_creation_year"] == 2026).sum()),
        "Modalités d'éducation": frame["education_level"].nunique(),
        "Heures trav. moy. des sans-emploi": round(frame.loc[no_job, "weekly_work_hours"].mean(), 1),
    })
display(pd.DataFrame(checks).set_index("Extrait"))

print("Modalités d'éducation dans B :")
print(raw_b["education_level"].value_counts().to_string())


### 1.2 Décision documentée

| Champ | Source retenue | Justification | Utilisé dans le modèle ? |
|---|---|---|---|
| `account_creation_year` | **A** | B contient 3 768 créations en 2026, après la dernière connexion (2025-01 → 2026-01) : impossible. | Non (descriptif / âge du compte seulement) |
| `education_level` | **B harmonisé** | A a replié 1 203 libellés `Secondary`, `High School`, `Bachelor` dans `PhD` : incohérent. B les conserve ; on normalise vers les 6 modalités canoniques. | Non (η² ≈ 0 sur le bien-être) |
| `employment_status`, `weekly_work_hours` | **A**, mais heures exclues | A donne ~40 h/semaine aux « Unemployed », B les reclasse « Not employed » à 0 h. Vérité non connaissable. Le champ n'a aucun pouvoir explicatif → on l'écarte du modèle. | Non |
| `relationship_status`, `has_children` | **A** | Divergence non arbitrable faute de source externe. Aucun lien avec le bien-être (η² ≈ 0) → choix sans effet sur les conclusions ; B sert de test de sensibilité. | Non |
| Variables de comportement, bien-être | **A** | Identiques dans les deux extraits. | Oui |

**Argument clé pour la soutenance :** les champs divergents sont précisément ceux dont le η² sur les trois indicateurs de bien-être est ≈ 0 (§3.3). Le choix A/B ne change donc **aucune conclusion** sur le lien usage ↔ bien-être ; il ne change que le descriptif socio-démographique.

**Champs écartés d'emblée :**
- `app_name` : constante (« Instagram »).
- `daily_steps_count` : quasi constante (amplitude relative < 10 %), donc aucune information.
- `user_engagement_score` : construit comme une intensité par minute (corr. 0,97 avec les interactions/minute) ; l'utiliser comme prédicteur serait circulaire. On reconstruit nos propres ratios.


In [ ]:
num = raw_a.select_dtypes(include=np.number)
quality = pd.DataFrame({
    "min": num.min(), "max": num.max(), "moyenne": num.mean(), "écart-type": num.std(),
})
quality["amplitude / |moyenne|"] = ((quality["max"] - quality["min"]) / (quality["moyenne"].abs() + 1e-9))
display(quality[quality["amplitude / |moyenne|"] < 0.20].round(2))

constant_cols = [c for c in raw_a.columns if raw_a[c].nunique(dropna=False) == 1]
print("Colonnes constantes :", constant_cols)

identity_gap = (raw_a["sessions_per_day"] * raw_a["average_session_length_minutes"]
                - raw_a["daily_active_minutes_instagram"]).abs()
print(f"Écart max 'sessions × durée de session − minutes actives' : {identity_gap.max():.6f} min")

n_floor = int((raw_a["daily_active_minutes_instagram"] == 5).sum())
print(f"Utilisateurs à exactement 5,0 min/jour : {n_floor:,} ({n_floor / len(raw_a):.1%})")
print("→ valeur plancher de censure, pas une mesure continue : à flaguer.")


## 2. Nettoyage et périmètre d'analyse

Le nettoyage reste **minimal et traçable** : on ne corrige que ce qui est démontrable. Les valeurs extrêmes plausibles (gros utilisateurs, gros comptes) sont conservées : les supprimer fabriquerait l'histoire qu'on cherche à mesurer.


In [ ]:
EDU_MAP = {"Secondary": "High school", "High School": "High school", "Bachelor": "Bachelor’s"}

df = raw_a.copy()
df["education_level"] = raw_b["education_level"].replace(EDU_MAP)   # récupère les 1 203 libellés écrasés
df["last_login_date"] = pd.to_datetime(df["last_login_date"])
df["is_minor"] = df["age"] < 18
df["account_age_years"] = 2026 - df["account_creation_year"]
df["usage_floor_flag"] = (df["daily_active_minutes_instagram"] == 5).astype(int)

DROP_COLS = ["app_name", "daily_steps_count", "weekly_work_hours"]
df = df.drop(columns=DROP_COLS)

print("Colonnes supprimées :", DROP_COLS)
print("\nÉducation après harmonisation :")
print(df["education_level"].value_counts().to_string())
print(f"\nMineurs (13–17 ans) : {int(df['is_minor'].sum()):,} ({df['is_minor'].mean():.1%})")
print(f"Utilisateurs à 5,0 min/jour : {int(df['usage_floor_flag'].sum()):,}")
print(f"Dimensions finales : {df.shape[0]:,} × {df.shape[1]}")


## 3. Exploration

### 3.1 Distributions

On regarde d'abord la forme des variables de bien-être et du temps d'usage : elle conditionne le choix des transformations et des modèles.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes[0, 0].hist(df["perceived_stress_score"], bins=40, color="#4C72B0")
axes[0, 0].set(title="Stress perçu (0–40)", xlabel="score", ylabel="nb users")
axes[0, 1].hist(df["self_reported_happiness"], bins=10, color="#55A868", rwidth=0.85)
axes[0, 1].set(title="Bonheur auto-déclaré (1–10)", xlabel="score", ylabel="nb users")
axes[1, 0].hist(df["sleep_hours_per_night"], bins=40, color="#C44E52")
axes[1, 0].set(title="Sommeil (heures/nuit)", xlabel="heures", ylabel="nb users")
axes[1, 1].hist(np.log1p(df["daily_active_minutes_instagram"]), bins=50, color="#8172B2")
axes[1, 1].set(title="log(1 + minutes actives/jour)", xlabel="log(minutes)", ylabel="nb users")
plt.tight_layout()
save_fig("01_distributions.png", dpi=120)
plt.show()

display(df[["perceived_stress_score", "self_reported_happiness", "sleep_hours_per_night",
             "daily_active_minutes_instagram"]].describe().T.round(2))


**Lecture.** Le stress est presque uniforme sur 0–40 ; le bonheur est uniforme sur 1–10. Ces deux variables ont donc été générées indépendamment l'une de l'autre (r ≈ 0). Le temps d'usage est étalé de 5 à 565 min avec un pic artificiel à 5,0 min.


### 3.2 Corrélations usage ↔ bien-être

On sépare bien-être (cibles) et usage (prédicteurs). `user_engagement_score` est gardé ici uniquement pour diagnostic, pas comme feature.


In [ ]:
wellbeing_cols = ["perceived_stress_score", "self_reported_happiness", "sleep_hours_per_night"]

usage_cols = [
    "daily_active_minutes_instagram", "sessions_per_day", "average_session_length_minutes",
    "posts_created_per_week", "reels_watched_per_day", "stories_viewed_per_day",
    "likes_given_per_day", "comments_written_per_day", "dms_sent_per_week", "dms_received_per_week",
    "ads_viewed_per_day", "time_on_feed_per_day", "time_on_explore_per_day",
    "time_on_messages_per_day", "time_on_reels_per_day", "followers_count", "following_count",
    "notification_response_rate", "user_engagement_score",
]

corr = df[wellbeing_cols + usage_cols].corr()
display(corr.loc[wellbeing_cols, usage_cols].T.round(2)
        .sort_values("perceived_stress_score", ascending=False))

# Heatmap lisible
hm_cols = wellbeing_cols + [
    "daily_active_minutes_instagram", "sessions_per_day", "average_session_length_minutes",
    "reels_watched_per_day", "stories_viewed_per_day", "likes_given_per_day",
    "comments_written_per_day", "dms_sent_per_week", "time_on_feed_per_day",
    "time_on_reels_per_day", "followers_count", "notification_response_rate",
]
hm = df[hm_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(hm.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(hm_cols))); ax.set_xticklabels(hm_cols, rotation=90, fontsize=8)
ax.set_yticks(range(len(hm_cols))); ax.set_yticklabels(hm_cols, fontsize=8)
for i in range(len(hm_cols)):
    for j in range(len(hm_cols)):
        ax.text(j, i, f"{hm.values[i, j]:.2f}", ha="center", va="center", fontsize=6)
ax.set_title("Corrélations (Pearson)")
fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
save_fig("02_correlation_matrix.png", dpi=120)
plt.show()


### 3.3 Le lien est-il linéaire ? Et quels liens sont nuls ?

Deux questions distinctes :
1. La relation au temps est-elle monotone ou saturante ? (utile pour choisir `log(minutes)` et pour le discours produit)
2. Quels indicateurs n'ont **aucun** lien avec l'usage ? Un lien nul est un résultat, pas un échec.


In [ ]:
binned = df.copy()
binned["usage_decile"] = pd.qcut(binned["daily_active_minutes_instagram"], 10,
                                 labels=False, duplicates="drop")
deciles = binned.groupby("usage_decile").agg(
    minutes=("daily_active_minutes_instagram", "mean"),
    stress=("perceived_stress_score", "mean"),
    happiness=("self_reported_happiness", "mean"),
    sleep=("sleep_hours_per_night", "mean"),
)
display(deciles.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(deciles["minutes"], deciles["stress"], marker="o", color="#C44E52", label="Stress perçu")
ax.set(xlabel="Minutes actives / jour (moyenne du décile)", ylabel="Stress moyen (0–40)")
ax2 = ax.twinx()
ax2.plot(deciles["minutes"], deciles["happiness"], marker="s", color="#55A868", label="Bonheur")
ax2.plot(deciles["minutes"], deciles["sleep"], marker="^", color="#4C72B0", label="Sommeil (h)")
ax2.set_ylabel("Bonheur (1–10) / Sommeil (h)")
ax.set_title("Bien-être moyen par décile de temps d'usage")
lines = ax.get_lines() + ax2.get_lines()
ax.legend(lines, [l.get_label() for l in lines], loc="center right", fontsize=8)
plt.tight_layout()
save_fig("03_bienetre_par_decile.png", dpi=120)
plt.show()


In [ ]:
def eta_squared(frame: pd.DataFrame, categorical: str, target: str) -> float:
    '''Part de variance de `target` expliquée par les modalités de `categorical`.'''
    grand_mean = frame[target].mean()
    grouped = frame.groupby(categorical, observed=True)[target]
    ss_between = ((grouped.mean() - grand_mean) ** 2 * grouped.size()).sum()
    ss_total = ((frame[target] - grand_mean) ** 2).sum()
    return float(ss_between / ss_total)

lifestyle = ["age", "exercise_hours_per_week", "hobbies_count", "social_events_per_month",
             "books_read_per_year", "volunteer_hours_per_month", "travel_frequency_per_year",
             "body_mass_index"]
categorical = ["gender", "country", "income_level", "employment_status", "education_level",
               "relationship_status", "has_children", "diet_quality", "smoking",
               "alcohol_frequency", "content_type_preference", "subscription_status"]

null_findings = pd.DataFrame([
    {"Test": "Stress vs bonheur", "Statistique": "r de Pearson",
     "Valeur": df["perceived_stress_score"].corr(df["self_reported_happiness"])},
    {"Test": "Sommeil vs usage", "Statistique": "|r| max",
     "Valeur": df[["sleep_hours_per_night"] + usage_cols].corr()
                  .loc["sleep_hours_per_night", usage_cols].abs().max()},
    {"Test": "Mode de vie vs bien-être", "Statistique": "|r| max",
     "Valeur": df[lifestyle + wellbeing_cols].corr()
                  .loc[lifestyle, wellbeing_cols].abs().max().max()},
    {"Test": "Variables catégorielles vs bien-être", "Statistique": "η² max",
     "Valeur": max(eta_squared(df, c, t) for c in categorical for t in wellbeing_cols)},
])
display(null_findings.assign(Valeur=lambda d: d["Valeur"].round(4)))


### 3.4 Constats de l'exploration

1. **Le temps d'usage est le signal dominant.** Stress : r ≈ 0,83 ; bonheur : r ≈ −0,37. La relation est monotone sur les déciles.
2. **Le sommeil est un lien nul.** Sa corrélation avec toutes les variables d'usage est ≈ 0, et les déciles de temps affichent 7,00 h partout. On ne peut donc pas affirmer que l'usage perturbe le sommeil dans ces données.
3. **Stress et bonheur sont quasi indépendants** (r ≈ 0). Ce n'est pas un résultat « psychologique » : c'est une propriété du générateur synthétique. Conséquence : un score composite qui les additionne est défendable, mais il ne faut pas parler de « santé mentale » comme d'un construit unique.
4. **Les variables socio-démographiques et de mode de vie n'expliquent rien** (η² ≈ 0). Toute segmentation construite sur ces champs serait du bruit.
5. **Le temps d'usage est presque le seul axe de variation.** C'est précisément le défi posé par le sujet : que dire de plus qu'un classement par temps ?

Ces constats seront re-testés avec l'extrait B en sensibilité (§9).


## 4. Feature engineering

Chaque variable dérivée répond à une question produit précise. On évite d'empiler des ratios redondants : `sessions × durée = minutes` à l'arrondi près (écart max observé : 2,4 min), donc garder les trois serait une colinéarité quasi parfaite.

| Variable | Question | Construction |
|---|---|---|
| `minutes_per_day` | intensité globale (modèle) | minutes actives / jour |
| `log_minutes` | intensité compressée (clustering) | `log1p(minutes/jour)` |
| `avg_session_min` | sessions longues ou courtes ? | durée moyenne d'une session |
| `feed_share`, `reels_share`, `msg_share` | **composition** du temps | temps par surface / somme des 4 surfaces (`explore` = référence implicite) |
| `interactions_per_min` | usage actif ou passif ? | (likes + commentaires + posts/7 + DM/7) / minutes |
| `passive_active_ratio` | consommation vs production | (reels + stories + pubs vues) / (actions actives + 1) |
| `follower_ratio` | position sociale | log(1+followers) − log(1+following) |
| `notification_response_rate` | réactivité | taux de réponse aux notifications |
| `account_age_years` | ancienneté | 2026 − année de création |

Les parts de temps somment à 1 : on en exclut une pour éviter une colinéarité parfaite dans les modèles linéaires.


In [ ]:
TIME_COLS = ["time_on_feed_per_day", "time_on_explore_per_day",
             "time_on_messages_per_day", "time_on_reels_per_day"]

def build_features(frame: pd.DataFrame) -> pd.DataFrame:
    '''Construit les variables dérivées à partir des champs disponibles pour TOUS les users.'''
    f = pd.DataFrame(index=frame.index)
    total_time = frame[TIME_COLS].sum(axis=1).replace(0, np.nan)

    f["minutes_per_day"] = frame["daily_active_minutes_instagram"]
    f["log_minutes"] = np.log1p(frame["daily_active_minutes_instagram"])
    f["avg_session_min"] = frame["average_session_length_minutes"]
    f["feed_share"] = frame["time_on_feed_per_day"] / total_time
    f["reels_share"] = frame["time_on_reels_per_day"] / total_time
    f["msg_share"] = frame["time_on_messages_per_day"] / total_time

    active_daily = (frame["likes_given_per_day"] + frame["comments_written_per_day"]
                    + frame["posts_created_per_week"] / 7 + frame["dms_sent_per_week"] / 7)
    f["interactions_per_min"] = active_daily / frame["daily_active_minutes_instagram"]

    passive = (frame["reels_watched_per_day"] + frame["stories_viewed_per_day"]
               + frame["ads_viewed_per_day"])
    f["passive_active_ratio"] = passive / (active_daily + 1)

    f["follower_ratio"] = np.log1p(frame["followers_count"]) - np.log1p(frame["following_count"])
    f["notification_response_rate"] = frame["notification_response_rate"]
    f["account_age_years"] = frame["account_age_years"]
    return f

features = build_features(df)
ALL_FEATURES = list(features.columns)
# Le modèle utilise les minutes brutes (relation linéaire avec le stress).
# Le clustering utilise la version log (échelle comparable aux parts de temps).
FEATURES = [c for c in ALL_FEATURES if c != "log_minutes"]
CLUSTER_FEATURES = [c for c in ALL_FEATURES if c != "minutes_per_day"]
display(features[ALL_FEATURES].describe().T.round(3))


In [ ]:
feature_audit = pd.DataFrame({
    "corr_stress": [features[c].corr(df["perceived_stress_score"]) for c in ALL_FEATURES],
    "corr_bonheur": [features[c].corr(df["self_reported_happiness"]) for c in ALL_FEATURES],
    "corr_sommeil": [features[c].corr(df["sleep_hours_per_night"]) for c in ALL_FEATURES],
}, index=ALL_FEATURES)
feature_audit["|r| max (stress/bonheur)"] = feature_audit[["corr_stress", "corr_bonheur"]].abs().max(axis=1)
display(feature_audit.round(3).sort_values("|r| max (stress/bonheur)", ascending=False))

# Redondance entre features
corr_abs = features[ALL_FEATURES].corr().abs()
upper = np.triu(np.ones(corr_abs.shape, dtype=bool), k=1)
top_pairs = (corr_abs.where(upper)
               .stack().sort_values(ascending=False).head(5))
print("Paires de features les plus corrélées :")
print(top_pairs.round(3).to_string())


### 4.1 Ces variables ajoutent-elles quelque chose au temps seul ?

Test simple et honnête : régression linéaire du stress et du bonheur, avec `minutes_per_day` seul puis avec toutes les features. Le **gain de R²** mesure l'apport des variables de composition et d'intensité.


In [ ]:
y_stress = df["perceived_stress_score"]
y_happy = df["self_reported_happiness"]

idx_train, idx_test = train_test_split(df.index, test_size=0.3, random_state=RANDOM_STATE)

def r2_for(target: pd.Series, cols: list[str]) -> float:
    model = LinearRegression().fit(features.loc[idx_train, cols], target.loc[idx_train])
    return r2_score(target.loc[idx_test], model.predict(features.loc[idx_test, cols]))

increment = pd.DataFrame({
    "R² temps seul": [r2_for(y_stress, ["minutes_per_day"]), r2_for(y_happy, ["minutes_per_day"])],
    "R² toutes features": [r2_for(y_stress, FEATURES), r2_for(y_happy, FEATURES)],
}, index=["Stress", "Bonheur"])
increment["Gain"] = increment["R² toutes features"] - increment["R² temps seul"]
display(increment.round(3))


## 5. Scores : bien-être et intensité d'usage

### 5.1 Score de bien-être

Choix assumé :
- **Stress inversé + bonheur**, chacun standardisé (`z`), donc poids égal.
- **Sommeil exclu** : son lien avec l'usage est nul, l'inclure ne ferait qu'ajouter du bruit et donnerait l'illusion de mesurer la santé.
- Score final ramené sur 0–100 pour la communication produit ; la cible binaire est le **médiane split** (interprétable, équilibré).

La variante incluant le sommeil sert de test de robustesse (corrélation de rang).


In [ ]:
def zscore(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std(ddof=0)

df["wellbeing_score"] = zscore(-df["perceived_stress_score"]) + zscore(df["self_reported_happiness"])
df["wellbeing_0_100"] = (100 * (df["wellbeing_score"] - df["wellbeing_score"].min())
                         / (df["wellbeing_score"].max() - df["wellbeing_score"].min()))
df["high_wellbeing"] = (df["wellbeing_score"] > df["wellbeing_score"].median()).astype(int)

score_with_sleep = (zscore(-df["perceived_stress_score"]) + zscore(df["self_reported_happiness"])
                    + zscore(df["sleep_hours_per_night"]))
rho = df["wellbeing_score"].rank().corr(score_with_sleep.rank())

print(f"Corrélation de rang entre le score principal et la variante avec sommeil : {rho:.3f}")
print("Répartition de la cible :")
print(df["high_wellbeing"].value_counts(normalize=True).round(3).to_string())

alt_weights = 0.7 * zscore(-df["perceived_stress_score"]) + 0.3 * zscore(df["self_reported_happiness"])
print(f"Corrélation de rang avec des poids 70/30 : {df['wellbeing_score'].rank().corr(alt_weights.rank()):.3f}")

sample_plot = df.sample(20_000, random_state=RANDOM_STATE)
fig, ax = plt.subplots(figsize=(9, 4))
hb = ax.hexbin(sample_plot["daily_active_minutes_instagram"], sample_plot["wellbeing_0_100"],
               gridsize=45, cmap="viridis", mincnt=1, yscale="linear")
ax.set(xlabel="Minutes actives / jour", ylabel="Score de bien-être (0–100)",
       title="Score de bien-être vs temps d'usage (échantillon de 20 000 users)")
fig.colorbar(hb, ax=ax, label="nb users")
plt.tight_layout()
save_fig("04_score_vs_temps.png", dpi=120)
plt.show()


### 5.2 Segmentation « produit » par seuils

Plutôt que de découper en déciles (illisibles pour le produit), on fixe quatre seuils métier : **< 30 min**, **30–120**, **120–240**, **> 240 min/jour**. Ils sont assignables à tout utilisateur sans modèle et produisent des segments de grande taille.


In [ ]:
SEGMENT_BINS = [0, 30, 120, 240, np.inf]
SEGMENT_LABELS = ["Usage très faible (<30 min)", "Usage modéré (30–120 min)",
                  "Usage élevé (120–240 min)", "Usage très élevé (>240 min)"]

df["usage_segment"] = pd.cut(df["daily_active_minutes_instagram"], bins=SEGMENT_BINS,
                             labels=SEGMENT_LABELS, right=False)

segment_profile = df.groupby("usage_segment", observed=True).agg(
    n=("user_id", "size"),
    minutes=("daily_active_minutes_instagram", "mean"),
    stress=("perceived_stress_score", "mean"),
    bonheur=("self_reported_happiness", "mean"),
    sommeil=("sleep_hours_per_night", "mean"),
    pct_haut_bienetre=("high_wellbeing", "mean"),
)
segment_profile["part"] = segment_profile["n"] / len(df)
display(segment_profile.round(3))


## 6. Modèle : prédire un score de bien-être élevé

**Cible :** `high_wellbeing` (médiane split).
**Prédicteurs :** uniquement des variables d'activité disponibles pour tous les users.
**Baseline obligatoire :** le temps seul, puisqu'on doit démontrer l'apport des autres variables.
**Protocole :** split stratifié 70/30, graine fixe. Pas de validation croisée imbriquée à ce stade : les résultats sont un premier point de repère, pas une estimation finale.


In [ ]:
X = features
y = df["high_wellbeing"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

models = {
    "Baseline — temps seul (logit)": (
        ["minutes_per_day"],
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    ),
    "Comportement complet (logit)": (
        FEATURES,
        make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    ),
    "Comportement complet (gradient boosting)": (
        FEATURES,
        HistGradientBoostingClassifier(max_iter=250, early_stopping=False,
                                       random_state=RANDOM_STATE),
    ),
}

results, roc_curves, fitted = [], {}, {}
for name, (cols, model) in models.items():
    model.fit(X_train[cols], y_train)
    proba = model.predict_proba(X_test[cols])[:, 1]
    results.append({
        "Modèle": name,
        "AUC": roc_auc_score(y_test, proba),
        "PR-AUC": average_precision_score(y_test, proba),
        "Brier": brier_score_loss(y_test, proba),
    })
    roc_curves[name] = roc_curve(y_test, proba)
    fitted[name] = model

results_df = pd.DataFrame(results).set_index("Modèle")
display(results_df.round(3))

fig, ax = plt.subplots(figsize=(7, 5))
for name, (fpr, tpr, _) in roc_curves.items():
    ax.plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, fitted[name].predict_proba(X_test[models[name][0]])[:, 1]):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set(xlabel="Taux de faux positifs", ylabel="Taux de vrais positifs", title="Courbes ROC")
ax.legend(fontsize=7, loc="lower right")
plt.tight_layout()
save_fig("05_roc.png", dpi=120)
plt.show()


In [ ]:
best_name = "Comportement complet (gradient boosting)"
best_model = fitted[best_name]

sample_idx = X_test.sample(5_000, random_state=RANDOM_STATE).index
perm = permutation_importance(
    best_model, X_test.loc[sample_idx, FEATURES], y_test.loc[sample_idx],
    n_repeats=3, random_state=RANDOM_STATE, scoring="roc_auc",
)
importance = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False)
display(importance.round(4).to_frame("Perte d'AUC si permuté"))

auc_time = results_df.loc["Baseline — temps seul (logit)", "AUC"]
auc_full = results_df.loc["Comportement complet (gradient boosting)", "AUC"]
print(f"Gain d'AUC du modèle complet vs temps seul : {auc_full - auc_time:+.3f}")
print("→ Un gain faible signifie que le temps d'usage contient déjà l'essentiel de l'information.")


## 7. Segmentation non supervisée : tentative et diagnostic

On teste la route « clustering » sur les variables de comportement. Le but n'est pas d'obtenir des clusters à tout prix : c'est de vérifier s'il existe **des profils naturels** au-delà de l'intensité.

Diagnostics :
- silhouette (séparabilité),
- taille et profil des clusters,
- **ARI** avec les segments par seuils (les clusters redisent-ils le temps ?),
- **η²** du bien-être expliqué par les clusters vs par les déciles de temps.

On retient `k = 4` pour rendre le diagnostic lisible à la jury. La courbe de silhouette indique déjà que le seul découpage réellement séparé est `k = 2` (« usage minimal » contre « le reste ») ; les valeurs suivantes servent à vérifier si une structure plus fine existe.


In [ ]:
CLUSTER_FEATURES = ["log_minutes", "avg_session_min", "feed_share", "reels_share",
                    "msg_share", "interactions_per_min", "passive_active_ratio", "follower_ratio"]

scaler_cluster = StandardScaler().fit(features[CLUSTER_FEATURES])
cluster_sample = features.sample(50_000, random_state=RANDOM_STATE)
Z_sample = scaler_cluster.transform(cluster_sample[CLUSTER_FEATURES])

diagnostics = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(Z_sample)
    diagnostics.append({
        "k": k,
        "inertie": km.inertia_,
        "silhouette": silhouette_score(Z_sample, km.labels_, sample_size=10_000,
                                       random_state=RANDOM_STATE),
    })
diagnostics = pd.DataFrame(diagnostics).set_index("k")
display(diagnostics.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(diagnostics.index, diagnostics["inertie"], marker="o")
axes[0].set(title="Inertie intra-cluster", xlabel="k")
axes[1].plot(diagnostics.index, diagnostics["silhouette"], marker="o", color="#C44E52")
axes[1].set(title="Silhouette (plus haut = mieux séparé)", xlabel="k")
plt.tight_layout()
save_fig("06_diagnostic_kmeans.png", dpi=120)
plt.show()


In [ ]:
K = 4
kmeans = KMeans(n_clusters=K, n_init=20, random_state=RANDOM_STATE).fit(Z_sample)
df["cluster"] = kmeans.predict(scaler_cluster.transform(features[CLUSTER_FEATURES]))

cluster_profile = df.groupby("cluster").agg(
    n=("user_id", "size"),
    minutes=("daily_active_minutes_instagram", "mean"),
    sessions=("sessions_per_day", "mean"),
    duree_session=("average_session_length_minutes", "mean"),
    part_feed=("time_on_feed_per_day", "sum"),
    stress=("perceived_stress_score", "mean"),
    bonheur=("self_reported_happiness", "mean"),
    sommeil=("sleep_hours_per_night", "mean"),
    pct_haut_bienetre=("high_wellbeing", "mean"),
)
cluster_profile["part"] = cluster_profile["n"] / len(df)
display(cluster_profile.drop(columns="part_feed").round(2))

ari_cluster_segments = adjusted_rand_score(df["cluster"], df["usage_segment"])
eta_cluster = eta_squared(df, "cluster", "wellbeing_score")
eta_segments = eta_squared(df, "usage_segment", "wellbeing_score")

df["usage_decile"] = pd.qcut(df["daily_active_minutes_instagram"], 10, labels=False, duplicates="drop")
eta_deciles = eta_squared(df, "usage_decile", "wellbeing_score")

print(f"ARI clusters KMeans vs segments par seuils : {ari_cluster_segments:.3f}")
print(f"Variance du score de bien-être expliquée — clusters KMeans : {eta_cluster:.3f}")
print(f"Variance du score de bien-être expliquée — segments par seuils : {eta_segments:.3f}")
print(f"Variance du score de bien-être expliquée — déciles de temps : {eta_deciles:.3f}")


### 7.1 Interprétation

Le clustering retrouve essentiellement **un petit groupe d'usage minimal** (les users à 5 min/jour) et laisse le reste en un continuum mal séparé : trois clusters sur quatre sont quasi identiques. Les silhouettes sont faibles (maximum à k=2, soit « faible usage » contre « le reste »), l'ARI avec les seuils de temps est très bas (les clusters ne retrouvent même pas un découpage temporel simple), et les clusters n'expliquent pas plus de variance de bien-être qu'un simple découpage du temps.

**Conclusion défendable :** il n'existe pas, dans ces données, de profils comportementaux naturels au-delà de l'intensité d'usage. La segmentation utile est donc **un découpage interprétable de l'intensité**, pas un cluster KMeans. C'est un résultat, et il répond directement à la question « qu'apporte votre segmentation par rapport au temps passé ? ».

### 7.2 Stabilité de la solution non supervisée

Même si on ne la retient pas, on mesure sa stabilité pour appuyer le diagnostic : deux moitiés de l'échantillon doivent retrouver les mêmes clusters.


In [ ]:
half = features.sample(40_000, random_state=RANDOM_STATE)
h1, h2 = half.iloc[:20_000], half.iloc[20_000:]

km1 = KMeans(n_clusters=K, n_init=10, random_state=1).fit(scaler_cluster.transform(h1[CLUSTER_FEATURES]))
km2 = KMeans(n_clusters=K, n_init=10, random_state=2).fit(scaler_cluster.transform(h2[CLUSTER_FEATURES]))

common = features.sample(10_000, random_state=7)
labels_1 = km1.predict(scaler_cluster.transform(common[CLUSTER_FEATURES]))
labels_2 = km2.predict(scaler_cluster.transform(common[CLUSTER_FEATURES]))
stability_ari = adjusted_rand_score(labels_1, labels_2)
print(f"ARI split-half : {stability_ari:.3f}  (1 = parfaitement stable, 0 = hasard)")


## 8. Assigner un nouvel utilisateur

Deux fonctions, correspondant aux deux routes :
- `assign_usage_segment(minutes)` : règle déterministe, aucune dépendance au modèle ;
- `assign_cluster(profile)` : pipeline complet (features → standardisation → KMeans).

Le profil de démonstration est **synthétique** (valeurs médianes plausibles), pas une ligne du jeu de données.


In [ ]:
def assign_usage_segment(minutes_per_day: float) -> str:
    for label, low, high in zip(SEGMENT_LABELS, SEGMENT_BINS[:-1], SEGMENT_BINS[1:]):
        if low <= minutes_per_day < high:
            return label
    return SEGMENT_LABELS[-1]


def assign_cluster(user_profile: dict) -> int:
    row = build_features(pd.DataFrame([user_profile]))
    return int(kmeans.predict(scaler_cluster.transform(row[CLUSTER_FEATURES]))[0])


new_user = {
    "daily_active_minutes_instagram": 90.0,
    "average_session_length_minutes": 12.0,
    "time_on_feed_per_day": 35.0,
    "time_on_explore_per_day": 15.0,
    "time_on_messages_per_day": 12.0,
    "time_on_reels_per_day": 28.0,
    "likes_given_per_day": 60.0,
    "comments_written_per_day": 18.0,
    "posts_created_per_week": 2.0,
    "dms_sent_per_week": 14.0,
    "reels_watched_per_day": 90.0,
    "stories_viewed_per_day": 55.0,
    "ads_viewed_per_day": 10.0,
    "followers_count": 800.0,
    "following_count": 600.0,
    "notification_response_rate": 0.5,
    "account_age_years": 6,
}

print("Profil synthétique de démonstration :")
print(f"  segment par seuils : {assign_usage_segment(new_user['daily_active_minutes_instagram'])}")
print(f"  cluster KMeans     : {assign_cluster(new_user)}")
print(f"  probabilité d'être 'haut bien-être' (modèle) : "
      f"{best_model.predict_proba(build_features(pd.DataFrame([new_user]))[FEATURES])[0, 1]:.2f}")


## 9. Évaluation contre les critères et test de sensibilité

On reprend les critères fixés en tête de notebook et on les renseigne avec les chiffres calculés. On vérifie aussi que le choix A/B sur les champs divergents ne change pas les conclusions : on rejoue les corrélations clés sur l'extrait B.


In [ ]:
# Sensibilité : mêmes liens sur l'extrait B, pour les champs identiques
common_cols = [c for c in raw_a.columns if c in raw_b.columns]
b_corr_stress = raw_b["daily_active_minutes_instagram"].corr(raw_b["perceived_stress_score"])
b_corr_happy = raw_b["daily_active_minutes_instagram"].corr(raw_b["self_reported_happiness"])
b_corr_sleep = raw_b["daily_active_minutes_instagram"].corr(raw_b["sleep_hours_per_night"])
a_corr_stress = df["daily_active_minutes_instagram"].corr(df["perceived_stress_score"])
a_corr_happy = df["daily_active_minutes_instagram"].corr(df["self_reported_happiness"])
a_corr_sleep = df["daily_active_minutes_instagram"].corr(df["sleep_hours_per_night"])

sensitivity = pd.DataFrame({
    "Extrait A": [a_corr_stress, a_corr_happy, a_corr_sleep],
    "Extrait B": [b_corr_stress, b_corr_happy, b_corr_sleep],
}, index=["r(minutes, stress)", "r(minutes, bonheur)", "r(minutes, sommeil)"])
display(sensitivity.round(4))

min_segment = int(segment_profile["n"].min())
criteria = pd.DataFrame([
    {"Critère": "Assignable à tout user", "Mesure": "fonction sur champs d'activité",
     "Résultat": "OK (2 routes)"},
    {"Critère": "Stable", "Mesure": "règle de seuils (déterministe)",
     "Résultat": "1.00 (par construction)"},
    {"Critère": "Stable", "Mesure": "ARI split-half (KMeans, non retenu)",
     "Résultat": f"{stability_ari:.2f}"},
    {"Critère": "Segments non triviaux", "Mesure": "taille du plus petit segment",
     "Résultat": f"{min_segment:,} users"},
    {"Critère": "Apport vs temps seul", "Mesure": "ΔAUC (HGB − temps seul)",
     "Résultat": f"{auc_full - auc_time:+.3f}"},
    {"Critère": "Apport vs temps seul", "Mesure": "ΔR² stress (toutes features − temps)",
     "Résultat": f"{increment.loc['Stress', 'Gain']:+.3f}"},
    {"Critère": "Interprétable", "Mesure": "nombre de features",
     "Résultat": f"{len(FEATURES)}"},
    {"Critère": "Choix des données", "Mesure": "écart max des corrélations A vs B",
     "Résultat": f"{(sensitivity['Extrait A'] - sensitivity['Extrait B']).abs().max():.4f}"},
])
display(criteria)


## 10. Limites et protection des données

### 10.1 Ce que les données ne permettent pas de dire
- **Causalité** : on mesure des associations dans un panneau transversal. Rien ne permet d'affirmer que l'usage *cause* le stress, ni l'inverse, ni qu'un troisième facteur (solitude, temps disponible) explique les deux.
- **Généralisation** : le bien-être n'est connu que pour des volontaires d'un panneau opt-in. Ils ne représentent pas les 200 000 users, ni *a fortiori* l'ensemble des utilisateurs Instagram.
- **Statut synthétique** : les données sont une réplique générée. Les régularités (stress uniforme, sommeil sans lien) sont des propriétés du générateur, pas des faits sur les utilisateurs réels.
- **Sommeil** : lien nul dans ces données ; on ne peut pas conclure « Instagram n'affecte pas le sommeil », seulement « ces données ne montrent aucun lien ».
- **Score composite** : les poids égaux sont un choix, pas une vérité. La corrélation de rang avec des poids 70/30 et avec le sommeil inclus est fournie pour montrer la robustesse.

### 10.2 Protection des données (UE/UK, dont mineurs)
- **Données de santé** : le stress et le bonheur auto-déclarés sont des données de santé au sens du RGPD (art. 9). Ils ne servent **jamais** de variables d'entrée à la segmentation : uniquement d'étiquettes d'évaluation sur le panneau.
- **Inférence** : un segment corrélé au stress peut révéler indirectement un état de santé. Placer un user dans un segment n'est donc pas anodin : il faut une base légale, une finalité compatible et une analyse d'impact (DPIA).
- **Mineurs** : 18 913 users ont 13–17 ans. Un ciblage publicitaire ou produit fondé sur le bien-être est exclu ; les segments ne doivent pas être utilisés pour profiler des mineurs.
- **Minimisation** : pas de champ santé, pas de contenu des messages, pas de données de localisation dans les features. On ne conserve que des agrégats comportementaux.
- **Petites cellules** : aucune statistique n'est publiée sur un segment de moins de 50 users. Vérification ci-dessous.
- **Pas de ligne individuelle** : ce notebook n'affiche aucun enregistrement, même anonymisé.


In [ ]:
privacy = df.groupby("usage_segment", observed=True).agg(
    n=("user_id", "size"),
    mineurs=("is_minor", "sum"),
)
privacy["part_mineurs"] = privacy["mineurs"] / privacy["n"]
display(privacy.assign(part_mineurs=lambda d: (d["part_mineurs"] * 100).round(2)))

print(f"Plus petit segment : {int(privacy['n'].min()):,} users (règle : ≥ 50)")
print("Champs utilisés dans le clustering :")
for col in CLUSTER_FEATURES:
    print("  -", col)
print("Champs de santé utilisés comme features : aucun")
print("Champs identifiants affichés : aucun")


## 11. Synthèse et prochaines étapes

### Ce que ce premier notebook établit
1. Un **audit reproductible** des deux extraits, avec une décision par champ et un test de sensibilité.
2. Une **chaîne de features** justifiée, sans fuite de données de santé.
3. Un **score de bien-être** et une cible binaire interprétables.
4. Un **modèle de prédiction** avec une basseline « temps seul » et la mesure honnête de son apport.
5. Une **segmentation par seuils** assignable, stable et sûre, après démonstration que KMeans n'apporte pas de structure supplémentaire.
6. Les **limites** et les contraintes RGPD.

### Ce qu'il reste à faire (prochaines itérations)
- [ ] Validation croisée répétée (5×5) et intervalles de confiance sur l'AUC.
- [ ] Calibration du modèle (courbe de fiabilité, régression isotonique) avant tout usage produit.
- [ ] Modèles séparés stress / bonheur, et test d'interactions (temps × composition).
- [ ] Test explicite de l'apport de la composition **à l'intérieur** de chaque décile de temps (appariement).
- [ ] Comparaison avec un modèle sur variables brutes pour vérifier que le feature engineering n'apporte pas que de la colinéarité.
- [ ] Stabilité des segments par bootstrap sur plusieurs graines.
- [ ] Note de 2 pages pour l'équipe juridique : finalités, base légale, DPIA, règles sur les mineurs.
- [ ] Préparer la slides de soutenance : 1 message par slide, la limite en dernière slide.


In [ ]:
print("=== RÉSULTATS CLÉS (calculés à l'exécution) ===")
print(f"Stress vs temps d'usage          : r = {a_corr_stress:.3f}")
print(f"Bonheur vs temps d'usage         : r = {a_corr_happy:.3f}")
print(f"Sommeil vs temps d'usage         : r = {a_corr_sleep:.3f}  → lien nul")
print(f"AUC temps seul                   : {auc_time:.3f}")
print(f"AUC comportement complet (HGB)   : {auc_full:.3f}  (Δ = {auc_full - auc_time:+.3f})")
print(f"R² stress, temps seul            : {increment.loc['Stress', 'R² temps seul']:.3f}")
print(f"R² stress, toutes features       : {increment.loc['Stress', 'R² toutes features']:.3f}")
print(f"η² bien-être — déciles de temps  : {eta_deciles:.3f}")
print(f"η² bien-être — segments par seuils: {eta_segments:.3f}")
print(f"Plus petit segment               : {min_segment:,} users")
print()
print("Message pour le jury : le temps d'usage porte l'essentiel du signal ;")
print("la valeur ajoutée d'une segmentation est la lisibilité produit et l'assignabilité,")
print("pas une meilleure prédiction. Toute affirmation causale ou sur le sommeil est exclue.")
